# 🌡️ Laboratorio: Validación Climática Walk-Forward (Valencia)

**Objetivo:** Demostrar la universalidad del motor SINDy aplicándolo a datos climáticos reales (Temperatura + Presión Atmosférica) de Valencia, España.

**Hipótesis:** Si el motor descubre leyes físicas reales y no patrones estadísticos espurios, su rendimiento en datos climáticos (que sí obedecen termodinámica) debería ser **superior** al observado en datos financieros.

**Fuente de Datos:** Open-Meteo Historical Weather API (gratuita, sin API key).

**Variables:** `Close` = Temperatura (°C), `Volume` = Presión Atmosférica (hPa).

**Modo del Motor:** `disable_returns=True, disable_norm=True` (valores físicos absolutos, sin log-retornos).

**Archivo de Salida:** `macro_backtest_climate_db.csv` (Anexión Segura / Checkpointing).

In [ ]:
import sys
import os
import time
import warnings
sys.path.append(os.path.abspath('..'))

# Ocultar warnings matemáticos durante la simulación masiva
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from src.ui.climate_loader import ClimateLoader
from src.ui.market_loader import MarketLoader
from src.quant_engine.sensor import SpectralAnalyzer
from src.quant_engine.blender import ContinuousBlender
from src.quant_engine.nervous import RegimeShiftDetector
from src.quant_engine.physics import PhysicsDiscoverer
from src.quant_engine.auto_tuner import CUSUMAutoTuner

### 1. Configuración de la Batería de Pruebas Climáticas
Definimos los escenarios temporales para Valencia. Cada escenario captura diferentes regímenes meteorológicos.

In [ ]:
# 1. Escenarios Temporales para Valencia
# Cada escenario tiene un propósito de validación distinto
scenarios = [
    {'name': '1y_completo', 'start': '2024-01-01', 'end': '2024-12-31', 'desc': 'Año completo (estacionalidad + ciclos diurnos)'},
    {'name': '6mo_verano', 'start': '2024-04-01', 'end': '2024-09-30', 'desc': 'Verano mediterráneo (régimen estable)'},
    {'name': '6mo_invierno', 'start': '2024-10-01', 'end': '2025-03-31', 'desc': 'Invierno (frentes atlánticos, más caótico)'},
    {'name': '3mo_estival', 'start': '2024-07-01', 'end': '2024-09-30', 'desc': 'Verano seco (régimen más determinista)'},
]

db_path = "macro_backtest_climate_db.csv"

# 2. Manejador de DB y Checkpointing (Idempotencia)
processed_configs = set()
if os.path.exists(db_path):
    df_existente = pd.read_csv(db_path)
    if not df_existente.empty:
        agrupado = df_existente.groupby(['Escenario']).size().reset_index()
        for _, row in agrupado.iterrows():
            processed_configs.add(row['Escenario'])

print(f"⚙️ Total Escenarios Climáticos Configurados: {len(scenarios)}")
print(f"📂 Escenarios ya procesados y guardados en DB: {len(processed_configs)}\n")

### 2. Motor de Ejecución Walk-Forward Climático
Ejecuta el Walk-Forward con control granular: progreso por iteración y checkpoint periódico al CSV cada 10 iteraciones.

In [ ]:
CHECKPOINT_CADA = 10  # Guardar al CSV cada N iteraciones

for scenario in scenarios:
    name = scenario['name']
    
    if name in processed_configs:
        print(f"⏭️ Omitiendo {name} - Ya existe en la base de datos.")
        continue
        
    print(f"\n🌡️ Iniciando simulación climática: {name}")
    print(f"   📋 {scenario['desc']}")
    
    # --- DESCARGA DE DATOS CLIMÁTICOS ---
    try:
        df_clima = ClimateLoader.load_climate_data(
            start_date=scenario['start'],
            end_date=scenario['end']
        )
    except Exception as e:
        print(f"❌ Error descargando datos climáticos: {e}")
        continue
        
    total_velas = len(df_clima)
    if total_velas < 250:
        print(f"⚠️ Omitiendo {name}: Historial demasiado corto ({total_velas} horas).")
        continue
    
    # --- PARÁMETROS WALK-FORWARD CLIMÁTICO ---
    VENTANA_INICIAL = 168   # 1 semana (7 × 24h). Captura ~7 ciclos diurnos.
    HORIZONTE = 72          # 3 días a futuro (72 horas)
    BLOQUES = 12            # Tramos de 6 horas cada uno
    SALTO = 72              # Avanzar 3 días entre iteraciones (evita saturación computacional)
    
    max_valid_start = total_velas - HORIZONTE
    total_iteraciones = (max_valid_start - VENTANA_INICIAL) // SALTO + 1
    
    print(f"   ► Horas Disponibles: {total_velas} | Salto: {SALTO}h | Predicción: {HORIZONTE}h ({HORIZONTE//24} días)")
    print(f"   ► Iteraciones Estimadas: {total_iteraciones} | Checkpoint cada {CHECKPOINT_CADA} iteraciones")
    
    # --- LOOP WALK-FORWARD CON CONTROL GRANULAR ---
    results_buffer = []  # Buffer temporal para checkpoint
    end_idx = VENTANA_INICIAL
    iter_count = 0
    t_start_global = time.time()
    
    while end_idx <= max_valid_start:
        iter_count += 1
        t_iter_start = time.time()
        
        df_slice = df_clima.iloc[:end_idx].copy()
        actual_values = df_clima.iloc[end_idx:end_idx+HORIZONTE]['Close'].values
        
        try:
            # 1. Auto-Tuner CUSUM (busca el mejor drift para el régimen actual)
            tuner = CUSUMAutoTuner(df_slice, disable_norm=True, disable_returns=True)
            best_drift, tuner_report = tuner.run_search()
            
            # 2. Reconstruir estado físico con el mejor parámetro
            log_returns, volumen_z, precio_raw, dt_val = MarketLoader.prepare_quant_input(
                df_slice, disable_norm=True, disable_returns=True
            )
            t = np.arange(len(log_returns), dtype=np.float64) * dt_val
            
            # Moldeado topológico
            sensor = SpectralAnalyzer()
            fft_results = sensor.analyze(np.column_stack((log_returns, volumen_z)), dt=dt_val)
            b = ContinuousBlender(tolerance=0.0050)
            b.fit(t, log_returns, fft_results[0]['periods'], 0)
            b.fit(t, volumen_z, fft_results[1]['periods'], 1)
            
            r_smooth, r_dot, _ = b.compute_continuous(0, t)
            v_smooth, v_dot, _ = b.compute_continuous(1, t)
            
            # 3. Detectar Quiebres
            detector = RegimeShiftDetector(threshold=5.0, drift=best_drift)
            rep = detector.detect(log_returns, r_smooth)
            shift_idx = rep['shift_indices']
            
            # 4. Régimen Vigente
            start_regime = shift_idx[-1] if len(shift_idx) > 0 else 0
            end_regime = len(t)
            if end_regime - start_regime < 15:
                start_regime = shift_idx[-2] if len(shift_idx) > 1 else 0
            
            # 5. Extracción Física
            disc = PhysicsDiscoverer(poly_degree=1)
            x_matrix = np.column_stack((r_smooth, v_smooth))
            x_dot_matrix = np.column_stack((r_dot, v_dot))
            
            physics_rep = disc.extract_equations(
                t=t[start_regime:end_regime], x=x_matrix[start_regime:end_regime],
                x_dot=x_dot_matrix[start_regime:end_regime], dt=dt_val,
                horizon_steps=HORIZONTE, sigma_res_r=0, sigma_res_v=0,
                last_price=precio_raw[-1], disable_norm=True, disable_returns=True
            )
            
            pred_values = physics_rep.get('prediction', {}).get('det_price_path', [])
            sindy_r2 = physics_rep.get('score', 0)
            
        except Exception as e:
            pred_values = []
            sindy_r2 = 0
            best_drift = 0
        
        # 6. Evaluación Vectorial vs Futuro Real
        row_metrics = {
            'Iteracion (Horas Vistas)': end_idx,
            'Drift (k)': best_drift,
            'SINDy R2': sindy_r2,
            'Validez': 'OK'
        }
        
        block_size = HORIZONTE // BLOQUES
        
        if len(pred_values) != HORIZONTE or (isinstance(physics_rep, dict) and physics_rep.get('empty_r_eq', False)):
            row_metrics['Validez'] = 'FALLO MATEMÁTICO'
        else:
            last_known = precio_raw[-1]
            for bi in range(BLOQUES):
                p_pred = np.array(pred_values[bi*block_size : (bi+1)*block_size])
                p_act = np.array(actual_values[bi*block_size : (bi+1)*block_size])
                
                # MAPE con epsilon para evitar división por ~0 en temperaturas cercanas a 0°C
                mape = np.mean(np.abs((p_act - p_pred) / np.maximum(np.abs(p_act), 0.1))) * 100
                
                # Naive Forecast (línea plana desde último valor conocido)
                p_naive = np.full_like(p_act, fill_value=last_known)
                naive_mape = np.mean(np.abs((p_act - p_naive) / np.maximum(np.abs(p_act), 0.1))) * 100
                
                # Hit Ratio Local (dirección dentro del bloque)
                delta_pred_local = np.sign(p_pred[-1] - p_pred[0])
                delta_act_local = np.sign(p_act[-1] - p_act[0])
                hit_local = 1.0 if delta_pred_local == delta_act_local else 0.0
                
                # Hit Ratio Acumulado (destino final vs origen)
                delta_pred_cum = np.sign(p_pred[-1] - last_known)
                delta_act_cum = np.sign(p_act[-1] - last_known)
                hit_cum = 1.0 if delta_pred_cum == delta_act_cum else 0.0
                
                row_metrics[f'MAPE_B{bi+1}'] = round(mape, 2)
                row_metrics[f'Naive_MAPE_B{bi+1}'] = round(naive_mape, 2)
                row_metrics[f'Hit_B{bi+1}'] = hit_local
                row_metrics[f'CumHit_B{bi+1}'] = hit_cum
        
        results_buffer.append(row_metrics)
        
        # --- PROGRESO ---
        t_iter_elapsed = time.time() - t_iter_start
        t_total_elapsed = time.time() - t_start_global
        avg_per_iter = t_total_elapsed / iter_count
        remaining = (total_iteraciones - iter_count) * avg_per_iter
        
        r2_str = f"{sindy_r2:.3f}" if sindy_r2 else "N/A"
        print(f"   [{iter_count}/{total_iteraciones}] Hora {end_idx} | R²={r2_str} | {t_iter_elapsed:.1f}s | ETA: {remaining/60:.1f}min")
        
        # --- CHECKPOINT PERIÓDICO ---
        if iter_count % CHECKPOINT_CADA == 0 and len(results_buffer) > 0:
            df_checkpoint = pd.DataFrame(results_buffer)
            df_checkpoint.insert(0, 'Total_Horas_Disponible', total_velas)
            df_checkpoint.insert(0, 'Fecha_Fin', scenario['end'])
            df_checkpoint.insert(0, 'Fecha_Inicio', scenario['start'])
            df_checkpoint.insert(0, 'Escenario', name)
            df_checkpoint.insert(0, 'Ciudad', 'Valencia')
            
            file_exists = os.path.isfile(db_path)
            df_checkpoint.to_csv(db_path, mode='a', header=not file_exists, index=False)
            print(f"   💾 Checkpoint: {len(results_buffer)} filas guardadas en {db_path}")
            results_buffer = []  # Vaciar buffer
        
        end_idx += SALTO
    
    # --- GUARDAR RESIDUO FINAL ---
    if len(results_buffer) > 0:
        df_final_batch = pd.DataFrame(results_buffer)
        df_final_batch.insert(0, 'Total_Horas_Disponible', total_velas)
        df_final_batch.insert(0, 'Fecha_Fin', scenario['end'])
        df_final_batch.insert(0, 'Fecha_Inicio', scenario['start'])
        df_final_batch.insert(0, 'Escenario', name)
        df_final_batch.insert(0, 'Ciudad', 'Valencia')
        
        file_exists = os.path.isfile(db_path)
        df_final_batch.to_csv(db_path, mode='a', header=not file_exists, index=False)
    
    total_time = time.time() - t_start_global
    print(f"\n✅ {name} completado: {iter_count} iteraciones en {total_time/60:.1f} minutos.")
    processed_configs.add(name)

### 3. Sanidad del Dataset
Lectura rápida para asegurar que la DB climática se está llenando correctamente.

In [ ]:
if os.path.exists(db_path):
    df_final = pd.read_csv(db_path)
    print(f"📊 Total Filas en la Base de Datos Climática: {len(df_final)}")
    print(f"📋 Escenarios registrados: {df_final['Escenario'].unique().tolist()}")
    print(f"\n--- Primeras filas ---")
    display(df_final.head())
    print(f"\n--- Últimas filas ---")
    display(df_final.tail())
else:
    print("El archivo CSV aún no ha sido creado.")